In [26]:
import os
import re
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import cohen_kappa_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

sns.set_theme(style="whitegrid")

# ============================================
# LOAD DATA
# ============================================
data_dir = "../data/" if os.path.exists("../data") else "data/"
os.makedirs(data_dir, exist_ok=True)
tsv_path = os.path.join(data_dir, "training_set_rel3.tsv")
csv_path = os.path.join(data_dir, "set1_essays.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
elif os.path.exists(tsv_path):
    try:
        df_all = pd.read_csv(tsv_path, sep="\t", encoding="utf-8")
    except UnicodeDecodeError:
        df_all = pd.read_csv(tsv_path, sep="\t", encoding="latin1")
    df = df_all[df_all["essay_set"] == 1][["essay_id", "essay", "domain1_score"]].copy()
    df = df.dropna()
    df.to_csv(csv_path, index=False)
else:
    raise FileNotFoundError(
        f"Place 'training_set_rel3.tsv' (ASAP dataset) or 'set1_essays.csv' in '{data_dir}' before running."
    )

print(f"Total Set 1 Essays Loaded: {len(df)}")
print(df.head())

# ============================================
# CLEAN + SPLIT (70/15/15)
# ============================================
df_clean = df[df["domain1_score"] != 3].copy()  # remove single score=3 outlier (too rare to stratify)

train_df, temp_df = train_test_split(
    df_clean, test_size=0.30, random_state=42, stratify=df_clean["domain1_score"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["domain1_score"]
)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")

# ============================================
# FEATURE ENGINEERING (no Java, no network calls)
# ============================================
def extract_features(data):
    word_counts, sentence_counts = [], []
    vocab_richnesses, avg_sentence_lengths = [], []
    grammar_errors_counts = []

    for essay in data["essay"]:
        text = str(essay)
        words = re.findall(r"\b[a-zA-Z']+\b", text.lower())
        w_count = len(words)
        word_counts.append(w_count)

        sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
        s_count = max(1, len(sentences))
        sentence_counts.append(s_count)

        vocab_richnesses.append(len(set(words)) / w_count if w_count > 0 else 0.0)
        avg_sentence_lengths.append(w_count / s_count if s_count > 0 else 0.0)

        # Lightweight grammar heuristic
        errors = 0
        for s in sentences:
            if s and not s[0].isupper():
                errors += 1
        errors += len(re.findall(r"\b(\w+)\s+\1\b", text.lower()))  # repeated words
        grammar_errors_counts.append(errors)

    return pd.DataFrame({
        "word_count": word_counts,
        "sentence_count": sentence_counts,
        "vocab_richness": vocab_richnesses,
        "avg_sentence_length": avg_sentence_lengths,
        "grammar_errors": grammar_errors_counts,
    }, index=data.index)

print("Extracting features...")
train_feat = extract_features(train_df)
val_feat = extract_features(val_df)
test_feat = extract_features(test_df)
print(train_feat.head())

# ============================================
# TF-IDF + FEATURE MATRICES
# ============================================
vectorizer = TfidfVectorizer(max_features=500, stop_words="english", ngram_range=(1, 2))

X_train_tfidf = pd.DataFrame(vectorizer.fit_transform(train_df["essay"]).toarray(), index=train_df.index)
X_val_tfidf = pd.DataFrame(vectorizer.transform(val_df["essay"]).toarray(), index=val_df.index)
X_test_tfidf = pd.DataFrame(vectorizer.transform(test_df["essay"]).toarray(), index=test_df.index)

X_train = pd.concat([train_feat, X_train_tfidf], axis=1)
X_val = pd.concat([val_feat, X_val_tfidf], axis=1)
X_test = pd.concat([test_feat, X_test_tfidf], axis=1)

X_train.columns = X_train.columns.astype(str)
X_val.columns = X_val.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

y_train = train_df["domain1_score"].values
y_val = val_df["domain1_score"].values
y_test = test_df["domain1_score"].values

print("Feature matrices built:", X_train.shape, X_val.shape, X_test.shape)

# ============================================
# EVALUATION METRIC
# ============================================
def qwk(y_true, y_pred):
    y_pred_rounded = np.clip(np.round(y_pred), y_true.min(), y_true.max()).astype(int)
    return cohen_kappa_score(y_true, y_pred_rounded, weights="quadratic")

# ============================================
# NAIVE BASELINE
# ============================================
mean_score = y_train.mean()
naive_preds = np.full(len(y_val), mean_score)
naive_qwk = qwk(y_val, naive_preds)
print(f"Naive Baseline QWK: {naive_qwk:.4f}")

# ============================================
# MODELS: RIDGE + RANDOM FOREST
# ============================================
ridge_model = Ridge(random_state=42)
ridge_model.fit(X_train.values, y_train)
ridge_val_qwk = qwk(y_val, ridge_model.predict(X_val.values))
print(f"Ridge Validation QWK: {ridge_val_qwk:.4f}")

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train.values, y_train)
rf_val_qwk = qwk(y_val, rf_model.predict(X_val.values))
print(f"Random Forest Validation QWK: {rf_val_qwk:.4f}")

# ============================================
# FINAL TEST EVALUATION (touch test set once)
# ============================================
chosen_name = "Ridge" if ridge_val_qwk >= rf_val_qwk else "Random Forest"
chosen_model = ridge_model if ridge_val_qwk >= rf_val_qwk else rf_model

test_qwk = qwk(y_test, chosen_model.predict(X_test.values))
print(f"Chosen Model: {chosen_name}")
print(f"Final Test QWK: {test_qwk:.4f}")
print(f"(Naive={naive_qwk:.4f}, Ridge val={ridge_val_qwk:.4f}, RF val={rf_val_qwk:.4f}, Test={test_qwk:.4f})")

Total Set 1 Essays Loaded: 1783
   essay_id                                              essay  domain1_score
0         1  Dear local newspaper, I think effects computer...              8
1         2  Dear @CAPS1 @CAPS2, I believe that using compu...              9
2         3  Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...              7
3         4  Dear Local Newspaper, @CAPS1 I have found that...             10
4         5  Dear @LOCATION1, I know having computers has a...              8
Train: 1247 | Validation: 267 | Test: 268
Extracting features...
      word_count  sentence_count  vocab_richness  avg_sentence_length  \
1042         336              28        0.532738            12.000000   
1657         363              15        0.495868            24.200000   
641          319              22        0.517241            14.500000   
1693         449              23        0.394209            19.521739   
304          464              29        0.355603            16.000000 

In [ ]:
# ============================================
# COMPUTE PERCENTILES & GENERATE FEEDBACK & EXPORT MODELS
# ============================================
import pickle

features = ["word_count", "sentence_count", "vocab_richness", "avg_sentence_length", "grammar_errors"]
percentiles = [10, 25, 75, 90]

# Compute percentiles from train_feat
thresholds = {}
print("Computed Percentiles from train_feat:")
for feat in features:
    values = train_feat[feat]
    p10 = np.percentile(values, 10)
    p25 = np.percentile(values, 25)
    p75 = np.percentile(values, 75)
    p90 = np.percentile(values, 90)
    thresholds[feat] = (p10, p25, p75, p90)
    print(f"{feat:20} -> 10th: {p10:6.2f} | 25th: {p25:6.2f} | 75th: {p75:6.2f} | 90th: {p90:6.2f}")

def generate_feedback(features_dict):
    feedback = []
    for feat in features:
        val = features_dict.get(feat, 0)
        p10, p25, p75, p90 = thresholds[feat]
        
        if feat == "grammar_errors":
            if val <= p10:
                bucket = "bottom 10%"
                classification = "good"
            elif val <= p25:
                bucket = "10-25%"
                classification = "good"
            elif val <= p75:
                bucket = "middle range"
                classification = "good"
            elif val <= p90:
                bucket = "75-90%"
                classification = "mild notes"
            else:
                bucket = "top 10%"
                classification = "notable"
        elif feat == "vocab_richness":
            if val <= p10:
                bucket = "bottom 10%"
                classification = "notable"
            elif val <= p25:
                bucket = "10-25%"
                classification = "mild notes"
            elif val <= p75:
                bucket = "middle range"
                classification = "good"
            elif val <= p90:
                bucket = "75-90%"
                classification = "good"
            else:
                bucket = "top 10%"
                classification = "good"
        else:
            if val <= p10:
                bucket = "bottom 10%"
                classification = "notable"
            elif val <= p25:
                bucket = "10-25%"
                classification = "mild notes"
            elif val <= p75:
                bucket = "middle range"
                classification = "good"
            elif val <= p90:
                bucket = "75-90%"
                classification = "mild notes"
            else:
                bucket = "top 10%"
                classification = "notable"
            
        feedback.append(f"{feat}: value={val:.2f} ({bucket}) -> {classification}")
    return feedback

# Test on one real essay's extracted features from test_feat
test_essay_idx = test_feat.index[0]
test_essay_features = test_feat.loc[test_essay_idx].to_dict()
print(f"\nFeedback for Test Essay (ID {test_essay_idx}):")
for line in generate_feedback(test_essay_features):
    print("-", line)

# Export model, vectorizer, and thresholds strictly to workspace root directory
out_dir = ".." if os.path.exists("../notebooks") else "."
with open(f"{out_dir}/model.pkl", "wb") as f:
    pickle.dump(chosen_model, f)
with open(f"{out_dir}/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
with open(f"{out_dir}/thresholds.pkl", "wb") as f:
    pickle.dump(thresholds, f)
print(f"Exported assets successfully to {out_dir}/")


In [28]:
# ============================================
# EVALUATE 3 REPRESENTATIVE TEST ESSAYS
# ============================================
# Find lowest, highest, and middle score essays in test_df
min_score = test_df["domain1_score"].min()
max_score = test_df["domain1_score"].max()
med_score = test_df["domain1_score"].median()

lowest_idx = test_df[test_df["domain1_score"] == min_score].index[0]
highest_idx = test_df[test_df["domain1_score"] == max_score].index[0]
middle_idx = test_df[test_df["domain1_score"] == med_score].index[0]

cases = [
    ("LOWEST SCORE ESSAY", lowest_idx),
    ("MIDDLE SCORE ESSAY", middle_idx),
    ("HIGHEST SCORE ESSAY", highest_idx)
]

for label, idx in cases:
    actual = test_df.loc[idx, "domain1_score"]
    
    # Get features and predict
    feat_row = X_test.loc[[idx]].values
    pred_raw = chosen_model.predict(feat_row)[0]
    pred_score = int(np.clip(np.round(pred_raw), 2, 12))
    
    # Generate feedback from extracted features
    feat_dict = test_feat.loc[idx].to_dict()
    feedback_list = generate_feedback(feat_dict)
    
    print(f"\n=== {label} (Index: {idx}) ===")
    print(f"Actual Human Score:      {actual}")
    print(f"Model Predicted Score:   {pred_score} (raw: {pred_raw:.2f})")
    print("Linguistic Feedback:")
    for line in feedback_list:
        print(f"  - {line}")



=== LOWEST SCORE ESSAY (Index: 542) ===
Actual Human Score:      2
Model Predicted Score:   2 (raw: 2.10)
Linguistic Feedback:
  - word_count: value=16.00 (bottom 10%) -> notable
  - sentence_count: value=2.00 (bottom 10%) -> notable
  - vocab_richness: value=0.81 (top 10%) -> notable
  - avg_sentence_length: value=8.00 (bottom 10%) -> notable
  - grammar_errors: value=0.00 (bottom 10%) -> good

=== MIDDLE SCORE ESSAY (Index: 985) ===
Actual Human Score:      8
Model Predicted Score:   8 (raw: 8.07)
Linguistic Feedback:
  - word_count: value=299.00 (middle range) -> good
  - sentence_count: value=21.00 (middle range) -> good
  - vocab_richness: value=0.44 (middle range) -> good
  - avg_sentence_length: value=14.24 (middle range) -> good
  - grammar_errors: value=0.00 (bottom 10%) -> good

=== HIGHEST SCORE ESSAY (Index: 1155) ===
Actual Human Score:      12
Model Predicted Score:   9 (raw: 9.33)
Linguistic Feedback:
  - word_count: value=396.00 (middle range) -> good
  - sentence_coun